In [17]:
import pandas as pd

train = pd.read_csv("../data/train_processed.csv", parse_dates=["time"])
test = pd.read_csv("../data/test_processed.csv", parse_dates=["time"])

print(train.columns.tolist())

['time', 'temperature_2m_mean (°C)', 'daylight_duration (s)', 'snowfall_sum (cm)', 'relative_humidity_2m_mean (%)', 'pressure_msl_mean (hPa)', 'temperature_2m_max (°C)', 'temperature_2m_min (°C)', 'cloud_cover_mean (%)', 'wind_speed_10m_max (km/h)', 'precipitation_sum (mm)', 'sunshine_duration (s)', 'longitude', 'latitude', 'elevation', 'temp_range', 'sunshine_ratio', 'target_temp_next_day', 'month', 'day_of_week', 'day_of_year_sin', 'day_of_year_cos', 'wind_dir_sin', 'wind_dir_cos']


In [18]:
train = train.drop(columns=["temperature_2m_mean (°C)"])
test = test.drop(columns=["temperature_2m_mean (°C)"])

In [19]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

feature_cols = [c for c in train.columns if c not in ["time", "target_temp_next_day"]]

X_train = train[feature_cols]
y_train = train["target_temp_next_day"]
X_test = test[feature_cols]
y_test = test["target_temp_next_day"]

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = mean_squared_error(y_test, predictions) ** 0.5
r2 = r2_score(y_test, predictions)

print(f"MAE: {mae:.2f}°C")
print(f"RMSE: {rmse:.2f}°C")
print(f"R²: {r2:.3f}")

MAE: 1.07°C
RMSE: 1.40°C
R²: 0.973


In [25]:
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=True)
print(importances)

longitude                        0.000424
elevation                        0.000455
month                            0.000549
snowfall_sum (cm)                0.000669
latitude                         0.000766
day_of_week                      0.000872
wind_dir_cos                     0.001621
sunshine_ratio                   0.001629
day_of_year_sin                  0.001655
temp_range                       0.001717
precipitation_sum (mm)           0.001930
wind_dir_sin                     0.002127
day_of_year_cos                  0.002148
relative_humidity_2m_mean (%)    0.002170
sunshine_duration (s)            0.002347
cloud_cover_mean (%)             0.002447
daylight_duration (s)            0.002486
wind_speed_10m_max (km/h)        0.003406
pressure_msl_mean (hPa)          0.004886
temperature_2m_min (°C)          0.409560
temperature_2m_max (°C)          0.556136
dtype: float64


In [26]:
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importances.head(10))

temperature_2m_max (°C)          0.556136
temperature_2m_min (°C)          0.409560
pressure_msl_mean (hPa)          0.004886
wind_speed_10m_max (km/h)        0.003406
daylight_duration (s)            0.002486
cloud_cover_mean (%)             0.002447
sunshine_duration (s)            0.002347
relative_humidity_2m_mean (%)    0.002170
day_of_year_cos                  0.002148
wind_dir_sin                     0.002127
dtype: float64


In [29]:
import numpy as np
test_raw = pd.read_csv("../data/test_processed.csv", parse_dates=["time"])

naive_predictions = test_raw["temperature_2m_mean (°C)"]
y_test = test_raw["target_temp_next_day"]

naive_mae = mean_absolute_error(y_test, naive_predictions)
naive_rmse = mean_squared_error(y_test, naive_predictions) ** 0.5
naive_r2 = r2_score(y_test, naive_predictions)

print(f"Naive MAE: {naive_mae:.2f}°C")
print(f"Naive RMSE: {naive_rmse:.2f}°C")
print(f"Naive R²: {naive_r2:.3f}")

Naive MAE: 0.92°C
Naive RMSE: 1.25°C
Naive R²: 0.979
